# MNQ FVG ML Strategy - Research Notebook

This notebook contains research and analysis for the Micro E-mini Nasdaq-100 Fair Value Gap detection strategy with machine learning prediction system.

## Strategy Overview

- **Asset**: Micro E-mini Nasdaq-100 (MNQ) futures
- **Timeframes**: Multi-timeframe analysis (1min, 5min, 15min, 30min, 1hour)
- **Signals**: Fair Value Gap (FVG) detection with ML prediction
- **Risk Management**: 2% stop loss, 4% take profit
- **Position Sizing**: Based on ML confidence

In [ ]:
# QuantConnect Research Environment
from QuantConnect import *
from QuantConnect.Data import *
from QuantConnect.Algorithm import *
from QuantConnect.Research import *
from QuantConnect.Python import *

# Set up research environment
qb = QuantBook()

# Add MNQ futures
mnq = qb.AddFuture(Futures.Indices.NASDAQ100Micro, Resolution.Minute)
mnq.SetFilter(TimeSpan.Zero, TimeSpan.FromDays(182))

print("MNQ Futures added for research")

In [ ]:
# Fetch historical data for analysis
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 3, 31)

history = qb.History([mnq.Symbol], start_date, end_date, Resolution.Minute)
print(f"Fetched {len(history)} data points")
print(history.head())

In [ ]:
# FVG Detection Functions
def detect_fvg_bullish(bars):
    """Detect bullish fair value gaps"""
    if len(bars) < 3:
        return None
    
    current = bars.iloc[-1]
    prev1 = bars.iloc[-2]
    prev2 = bars.iloc[-3]
    
    # Bullish FVG: prev2.High < prev1.Low < current.Low
    if prev2['high'] < prev1['low'] and prev1['low'] < current['low']:
        return {
            'type': 'bullish',
            'top': prev2['high'],
            'bottom': prev1['low'],
            'time': current.name,
            'strength': abs(prev2['high'] - prev1['low']) / prev2['high']
        }
    return None

def detect_fvg_bearish(bars):
    """Detect bearish fair value gaps"""
    if len(bars) < 3:
        return None
    
    current = bars.iloc[-1]
    prev1 = bars.iloc[-2]
    prev2 = bars.iloc[-3]
    
    # Bearish FVG: prev2.Low > prev1.High > current.High
    if prev2['low'] > prev1['high'] and prev1['high'] > current['high']:
        return {
            'type': 'bearish',
            'top': prev1['high'],
            'bottom': prev2['low'],
            'time': current.name,
            'strength': abs(prev1['high'] - prev2['low']) / prev2['low']
        }
    return None

In [ ]:
# Analyze FVGs in the historical data
fvg_signals = []
window_size = 3

for i in range(window_size, len(history)):
    window = history.iloc[i-window_size:i]
    
    bullish_fvg = detect_fvg_bullish(window)
    if bullish_fvg:
        fvg_signals.append(bullish_fvg)
    
    bearish_fvg = detect_fvg_bearish(window)
    if bearish_fvg:
        fvg_signals.append(bearish_fvg)

print(f"Detected {len(fvg_signals)} FVG signals")

# Convert to DataFrame for analysis
if fvg_signals:
    import pandas as pd
    fvg_df = pd.DataFrame(fvg_signals)
    print("\nFVG Statistics:")
    print(fvg_df['type'].value_counts())
    print(f"\nAverage strength: {fvg_df['strength'].mean():.4f}")
    print(fvg_df.head())

In [ ]:
# Analyze FVG fill rates
def analyze_fvg_fill_rate(fvg_signals, price_data, lookforward_periods=[5, 10, 20]):
    """Analyze how often FVGs get filled within different time periods"""
    results = {}
    
    for periods in lookforward_periods:
        filled_count = 0
        total_count = len(fvg_signals)
        
        for fvg in fvg_signals:
            fvg_time = fvg['time']
            future_data = price_data.loc[fvg_time:].head(periods + 1)
            
            if len(future_data) > periods:
                if fvg['type'] == 'bullish':
                    # Check if price fills the gap from above
                    if any(future_data['low'] <= fvg['bottom']):
                        filled_count += 1
                else:  # bearish
                    # Check if price fills the gap from below
                    if any(future_data['high'] >= fvg['top']):
                        filled_count += 1
        
        fill_rate = filled_count / total_count if total_count > 0 else 0
        results[f'{periods}_periods'] = {
            'filled': filled_count,
            'total': total_count,
            'fill_rate': fill_rate
        }
    
    return results

if fvg_signals:
    fill_analysis = analyze_fvg_fill_rate(fvg_signals, history)
    print("FVG Fill Rate Analysis:")
    for period, stats in fill_analysis.items():
        print(f"{period}: {stats['filled']}/{stats['total']} ({stats['fill_rate']:.1%})")

In [ ]:
# Feature Engineering for ML
def extract_features(fvg, price_data, current_price):
    """Extract features for ML prediction"""
    features = {
        'gap_size': abs(fvg['top'] - fvg['bottom']) / fvg['top'],
        'distance_from_price': abs(current_price - (fvg['top'] + fvg['bottom']) / 2) / current_price,
        'strength': fvg['strength'],
        'hour_of_day': fvg['time'].hour / 24,
        'day_of_week': fvg['time'].dayofweek / 7,
        'is_bullish': 1 if fvg['type'] == 'bullish' else 0,
    }
    
    # Add recent price action features
    recent_data = price_data.loc[:fvg['time']].tail(20)
    if len(recent_data) > 1:
        features['volatility'] = recent_data['close'].pct_change().std()
        features['trend'] = (recent_data['close'].iloc[-1] - recent_data['close'].iloc[0]) / recent_data['close'].iloc[0]
    else:
        features['volatility'] = 0
        features['trend'] = 0
    
    return features

# Extract features for all FVGs
if fvg_signals:
    features_list = []
    for fvg in fvg_signals:
        current_price = history.loc[fvg['time'], 'close'] if fvg['time'] in history.index else None
        if current_price:
            features = extract_features(fvg, history, current_price)
            features_list.append(features)
    
    if features_list:
        features_df = pd.DataFrame(features_list)
        print("Feature Statistics:")
        print(features_df.describe())

In [ ]:
# Simple ML Model Training (for demonstration)
if fvg_signals and len(features_list) > 10:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, classification_report
    
    # Create labels (1 if FVG gets filled within 10 periods, 0 otherwise)
    labels = []
    for fvg in fvg_signals:
        fvg_time = fvg['time']
        future_data = history.loc[fvg_time:].head(11)
        
        filled = False
        if len(future_data) > 10:
            if fvg['type'] == 'bullish':
                filled = any(future_data['low'] <= fvg['bottom'])
            else:
                filled = any(future_data['high'] >= fvg['top'])
        
        labels.append(1 if filled else 0)
    
    # Train model
    X = pd.DataFrame(features_list)
    y = pd.Series(labels)
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    print(f"Model Accuracy: {accuracy:.3f}")
    print("\nFeature Importances:")
    for feature, importance in zip(X.columns, model.feature_importances_):
        print(f"{feature}: {importance:.3f}")

## Backtesting Results Summary

The main algorithm will be backtested using the QuantConnect platform with the following parameters:

- **Period**: 2024-01-01 to 2024-12-31
- **Initial Capital**: $100,000
- **Resolution**: Minute
- **Asset**: MNQ futures (Micro E-mini Nasdaq-100)

### Key Strategy Components:

1. **Multi-timeframe FVG Detection**: Analyzes 5 timeframes simultaneously
2. **ML Prediction**: Uses trained models to predict fill probability
3. **Confluence Scoring**: Prioritizes FVGs appearing in multiple timeframes
4. **Risk Management**: 2% stop loss, 4% take profit
5. **Position Sizing**: Scales positions based on ML confidence

### Expected Performance:

Based on historical analysis, we expect:
- FVG fill rate: ~60-70% within 10 periods
- Average trade duration: 15-30 minutes
- Win rate: ~55-60% with proper risk management
- Sharpe ratio: Target > 1.0

In [ ]:
# Save research results for reference
research_summary = {
    'data_period': f'{start_date.date()} to {end_date.date()}',
    'total_fvgs_detected': len(fvg_signals),
    'bullish_fvgs': len([f for f in fvg_signals if f['type'] == 'bullish']),
    'bearish_fvgs': len([f for f in fvg_signals if f['type'] == 'bearish']),
    'average_strength': sum(f['strength'] for f in fvg_signals) / len(fvg_signals) if fvg_signals else 0,
}

if 'fill_analysis' in locals():
    research_summary['fill_rates'] = fill_analysis

print("Research Summary:")
for key, value in research_summary.items():
    print(f"{key}: {value}")